In [79]:
from math import *

def sec_ratio(v) -> float:
    return (sqrt(4 * v - 3) - 1) / 2
    
def sec_ratio_2(v) -> float:
    return (sqrt(4 * v - 3) + 1) / 2
    
def sec_ratio_lowest(v) -> float:
    return (sqrt(4 * v - 7) + 1) / 2

def n_tiles(v: int, h: float) -> int:
    return v * ((v - 1) ** h - 1)/(v - 2) + 1

def n_tiles_at(v: int, h: int) -> int:
    return v * (v - 1) ** (h - 1)

In [80]:
n_tiles(4, 5), n_tiles_at(4, 5), f"on cover: {n_tiles(4, 5)*0.5 + 0.5} ({7*(n_tiles(4, 5)*0.5+0.5) - 3} chains)", f"complete tiling sigma n_1: {7*(n_tiles(4, 5))}", f"better than simplex ratio: {n_tiles(4, 5) / (4+1)}x"

(485.0,
 324,
 'on cover: 243.0 (1698.0 chains)',
 'complete tiling sigma n_1: 3395.0',
 'better than simplex ratio: 97.0x')

In [4]:
depth = 6
n_tiles(4, depth), n_tiles_at(4, depth), f"on cover: {n_tiles(4, depth)*0.5 + 0.5} ({7*(n_tiles(4, depth)*0.5+0.5) - 3} chains)", f"complete tiling sigma n_1: {7*(n_tiles(4, depth))}", f"better than simplex ratio: {n_tiles(4, depth) / (4+1)}x"

(1457.0,
 972,
 'on cover: 729.0 (5100.0 chains)',
 'complete tiling sigma n_1: 10199.0',
 'better than simplex ratio: 291.4x')

In [5]:
sec_ratio(3), sec_ratio(4)

(1.0, 1.3027756377319946)

In [6]:
sec_ratio_2(1.1), sec_ratio_2(1.5), sec_ratio_2(2), sec_ratio_2(3), sec_ratio_2(4), sec_ratio_2(5), sec_ratio_2(6), sec_ratio_2(7)

(1.0916079783099617,
 1.3660254037844386,
 1.618033988749895,
 2.0,
 2.302775637731995,
 2.5615528128088303,
 2.79128784747792,
 3.0)

In [7]:
def only_whole(ir):
    i, r = ir
    return floor(r) - r == 0
r_pairs = list((i,sec_ratio_2(i)) for i in range(2,100))
whole_rs = list(filter(only_whole, r_pairs))
whole_rs

[(3, 2.0),
 (7, 3.0),
 (13, 4.0),
 (21, 5.0),
 (31, 6.0),
 (43, 7.0),
 (57, 8.0),
 (73, 9.0),
 (91, 10.0)]

In [8]:
phi = r_pairs[0][1]
silver = (sqrt(2) + 1)/2
phi + silver
sec_ratio_2(1)

1.0

In [9]:
rs_z = list(zip(whole_rs[:-1], whole_rs[1:]))
r_diffs = list((ra, rb, b - a) for ((a, ra), (b, rb)) in rs_z)
r_diffs

[(2.0, 3.0, 4),
 (3.0, 4.0, 6),
 (4.0, 5.0, 8),
 (5.0, 6.0, 10),
 (6.0, 7.0, 12),
 (7.0, 8.0, 14),
 (8.0, 9.0, 16),
 (9.0, 10.0, 18)]

In [10]:
for v in range(3,6):
    r = sec_ratio_2(v)
    print(f"[v:{v}] LHS >= RHS: {r + r*r, v - 1} | r={r}")

[v:3] LHS >= RHS: (6.0, 2) | r=2.0
[v:4] LHS >= RHS: (7.6055512754639905, 3) | r=2.302775637731995
[v:5] LHS >= RHS: (9.123105625617661, 4) | r=2.5615528128088303


In [14]:
for v in range(3, 6):
    """Calculate tiling security buffer as a % of the lower bound.
    hypothesis: in essence tx fees can be up-to this % of the block reward without threatening tiling's security.
    Note: this assumes that we set relative chain-work between tiles to the lower threshhold.
    Whether this is possible or not may depend on things like the RAA response time."""
    r_lower = sec_ratio_2(v)
    r_upper = v-1
    d = r_upper - r_lower
    print(f"[v:{v}] LHS > RHS: {r_upper, r_lower} | diff: {d:.2f} | % of RHS: {d / r_lower * 100:.2f}%")


[v:3] LHS > RHS: (2, 2.0) | diff: 0.00 | % of RHS: 0.00%
[v:4] LHS > RHS: (3, 2.302775637731995) | diff: 0.70 | % of RHS: 30.28%
[v:5] LHS > RHS: (4, 2.5615528128088303) | diff: 1.44 | % of RHS: 56.16%


In [3]:
# NEW TILING STUFF 2022-03-17

from math import log
96*log(2)/log(3)

60.5692563428599

In [74]:
POW_BITS_EXCAP = 96
def layers_for(r: float, bits = POW_BITS_EXCAP):
    l = floor(bits * log(2) / log(r))
    return l

def capacity_mult(v: int, r: float):
    '''How many times greater than a single simplex at base layer. assumes 1 root tile'''
    h = (layers_for(r))
    nt = n_tiles(v, h)
    # divide the number of tiles by the valence b/c each maximal simplex will only have 1/v capacity for local chains
    return nt / v

def min_r_for_v(v):
    '''
    note re q threshold
    '''
    return 2 * v - 1

In [75]:
def f(v):
    r = min_r_for_v(v)
    return (capacity_mult(v, r), v, r, layers_for(r))
mult_v_pairs = list(map(f, range(3,17)))
mult_v_pairs.sort()
mvs_ordered = mult_v_pairs[::-1]


In [76]:
print("| v | r | d | mult |")
print("|---|---|---|------|")
print("\n".join(f"| {mv[1]} | {mv[2]} | {mv[3]} | {mv[0]:03e} |" for mv in mvs_ordered))

| v | r | d | mult |
|---|---|---|------|
| 16 | 31 | 10 | 4.118931e+10 |
| 15 | 29 | 10 | 2.225036e+10 |
| 14 | 27 | 10 | 1.148821e+10 |
| 13 | 25 | 10 | 5.628851e+09 |
| 12 | 23 | 10 | 2.593742e+09 |
| 11 | 21 | 10 | 1.111111e+09 |
| 10 | 19 | 10 | 4.358481e+08 |
| 9 | 17 | 10 | 1.533917e+08 |
| 8 | 15 | 10 | 4.707921e+07 |
| 7 | 13 | 10 | 1.209324e+07 |
| 6 | 11 | 10 | 2.441406e+06 |
| 5 | 9 | 10 | 3.495252e+05 |
| 4 | 7 | 10 | 2.952425e+04 |
| 3 | 5 | 10 | 1.023333e+03 |


In [77]:
def q_lt_safe_region(v):
    r = min_r_for_v(v)
    top = 1 + v/r + v*(v-1)/(r**2)
    bot = 1 + v/(r-v+1)
    return 0.5 * top / bot

def fmt_row(v):
    r = min_r_for_v(v)
    q = q_lt_safe_region(v)
    d = layers_for(r)
    m = capacity_mult(v,r)
    return f"| {q:.03f} | {v} | {r} | {d} |{m:9.3e} | {m/d:9.3e} |"

cols = ['>q', 'v', 'r', 'depth', 'mult', 'mult / depth']
print("| " + " | ".join(cols) + " |")
print("|" + "|".join(['---'] * len(cols)) + "|")
print("\n".join(fmt_row(v) for v in range(3,1024)))

| >q | v | r | depth | mult | mult / depth |
|---|---|---|---|---|---|
| 0.460 | 3 | 5 | 10 |1.023e+03 | 1.023e+02 |
| 0.454 | 4 | 7 | 10 |2.952e+04 | 2.952e+03 |
| 0.451 | 5 | 9 | 10 |3.495e+05 | 3.495e+04 |
| 0.448 | 6 | 11 | 10 |2.441e+06 | 2.441e+05 |
| 0.447 | 7 | 13 | 10 |1.209e+07 | 1.209e+06 |
| 0.446 | 8 | 15 | 10 |4.708e+07 | 4.708e+06 |
| 0.445 | 9 | 17 | 10 |1.534e+08 | 1.534e+07 |
| 0.444 | 10 | 19 | 10 |4.358e+08 | 4.358e+07 |
| 0.443 | 11 | 21 | 10 |1.111e+09 | 1.111e+08 |
| 0.443 | 12 | 23 | 10 |2.594e+09 | 2.594e+08 |
| 0.442 | 13 | 25 | 10 |5.629e+09 | 5.629e+08 |
| 0.442 | 14 | 27 | 10 |1.149e+10 | 1.149e+09 |
| 0.442 | 15 | 29 | 10 |2.225e+10 | 2.225e+09 |
| 0.441 | 16 | 31 | 10 |4.119e+10 | 4.119e+09 |
| 0.441 | 17 | 33 | 10 |7.330e+10 | 7.330e+09 |
| 0.441 | 18 | 35 | 10 |1.260e+11 | 1.260e+10 |
| 0.441 | 19 | 37 | 10 |2.100e+11 | 2.100e+10 |
| 0.441 | 20 | 39 | 10 |3.406e+11 | 3.406e+10 |
| 0.441 | 21 | 41 | 10 |5.389e+11 | 5.389e+10 |
| 0.440 | 22 | 43 | 10 |8.3

In [81]:
n_tiles(3, 10) * 2/3

2046.6666666666667

In [82]:
2000/4

500.0